In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the dataset
train = pd.read_csv('csv/train.csv')
test = pd.read_csv('csv/test.csv')

# 문제 정의 & 평가 기준 파악


## 문제 정의

### About the data


이 데이터셋은 실제 은행 계좌 개설 사기 탐지를 위한 대규모 실거래 데이터를 기반으로 생성된 데이터입니다.
100만 건의 계좌 개설 신청 데이터를 포함하고 있으며, 높은 클래스 불균형 등 현업 환경에서 흔히 직면하는 난제를 그대로 재현합니다.

### Files

    train.csv - the training set
    test.csv - the test set
    sample_submission.csv - a sample submission file in the correct format


### Columns


yearly_income
연간 소득 (10분위 값; 0.1=하위 10%, 0.9=상위 10%)

name_email_sim
이름-이메일 텍스트 유사도 (0~1)

prev_addr_months
이전 주소 거주 기간 (개월)

curr_addr_months
현재 주소 거주 기간 (개월)

age_bucket
나이 (10년 단위 반올림)

days_since_req
신청 후 경과 일수

init_transfer_amt
최초 이체 금액

payment_type
결제 방식 (5가지 코드)

zip_req_count_4w
최근 4주 동일 우편번호 신청 수

req_rate_6h
최근 6시간 시간당 평균 신청 수

req_rate_24h
최근 24시간 시간당 평균 신청 수

req_rate_4w
최근 4주 시간당 평균 신청 수

branch_req_count_8w
최근 8주 지점 신청 수

dob_email_count_4w
동일 생년월일(dob) 기준 이메일 수 (최근 4주)

employment_status
고용 상태 (7가지 코드)

credit_risk_score
신용 위험 점수 (높을수록 위험)

is_free_email
무료 이메일 여부

housing_status
주거 형태 (7가지 코드)

is_home_phone_valid
집 전화 유효 여부

is_mobile_valid
휴대폰 유효 여부

bank_months_count
계좌 개설 이후 경과 개월 수

has_other_cards
동일 은행 카드 보유 여부

req_credit_limit
요청 신용 한도

is_foreign_req
해외 요청 여부

application_source
신청 경로 (INTERNET / TELEAPP)

session_length_min
세션 시간 (분)

device_os
기기 운영체제

is_session_persistent
세션 유지 여부

device_email_cnt_8w
기기 기준 이메일 수 (최근 8주)

device_fraud_history
해당 기기의 과거 사기 이력

month_idx
신청 월 (0~7, 익명화)

fraud_bool
타겟 변수 (1 = 사기, 0 = 정상)


## train/test 데이터

In [2]:
TARGET = "fraud_bool"
ID_COL = "id"
print(train.shape, test.shape)
print(train.columns)

(700000, 33) (300000, 32)
Index(['id', 'fraud_bool', 'yearly_income', 'name_email_sim',
       'prev_addr_months', 'curr_addr_months', 'age_bucket', 'days_since_req',
       'init_transfer_amt', 'payment_type', 'zip_req_count_4w', 'req_rate_6h',
       'req_rate_24h', 'req_rate_4w', 'branch_req_count_8w',
       'dob_email_count_4w', 'employment_status', 'credit_risk_score',
       'is_free_email', 'housing_status', 'is_home_phone_valid',
       'is_mobile_valid', 'bank_months_count', 'has_other_cards',
       'req_credit_limit', 'is_foreign_req', 'application_source',
       'session_length_min', 'device_os', 'is_session_persistent',
       'device_email_cnt_8w', 'device_fraud_history', 'month_idx'],
      dtype='str')



## 평가 기준
Average Precision (AP)

제출한 결과물은 Precision–Recall 곡선의 Average Precision(AP)으로 평가됩니다.
AP는 모델이 양성 샘플을 얼마나 정확히 식별하는지를 종합적으로 측정하는 지표이며, 다음과 같이 계산됩니다:

여기서

$P_n$ : n번째 임계값에서의 Precision <br>
$R_n$ : n번째 임계값에서의 Recall <br>
$R_0$ = 0 으로 정의

# Load Data

In [3]:
train.head()

,id,fraud_bool,yearly_income,name_email_sim,prev_addr_months,curr_addr_months,age_bucket,days_since_req,init_transfer_amt,payment_type,...,has_other_cards,req_credit_limit,is_foreign_req,application_source,session_length_min,device_os,is_session_persistent,device_email_cnt_8w,device_fraud_history,month_idx
0,0,0,0.8,0.996036,9,19,40,0.019167,-0.795951,AB,...,1,510.0,0,INTERNET,4.125364,other,1,1,0,6
1,1,0,0.8,0.520470,12,10,30,0.010994,-0.667975,AD,...,0,1500.0,0,INTERNET,8.147967,macintosh,0,1,0,4
2,2,0,0.5,0.792133,162,11,40,70.144985,-1.051204,AC,...,0,200.0,0,INTERNET,4.996239,linux,1,1,0,3
3,3,0,0.1,0.846394,24,7,50,0.005721,-1.404540,AB,...,0,200.0,0,INTERNET,5.860972,linux,1,1,0,6
4,4,0,0.9,0.232846,-1,81,40,0.023110,-0.519206,AD,...,0,200.0,0,INTERNET,12.764765,windows,0,1,0,4


# EDA

In [4]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 33 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   id                     700000 non-null  int64  
 1   fraud_bool             700000 non-null  int64  
 2   yearly_income          700000 non-null  float64
 3   name_email_sim         700000 non-null  float64
 4   prev_addr_months       700000 non-null  int64  
 5   curr_addr_months       700000 non-null  int64  
 6   age_bucket             700000 non-null  int64  
 7   days_since_req         700000 non-null  float64
 8   init_transfer_amt      700000 non-null  float64
 9   payment_type           700000 non-null  str    
 10  zip_req_count_4w       700000 non-null  int64  
 11  req_rate_6h            700000 non-null  float64
 12  req_rate_24h           700000 non-null  float64
 13  req_rate_4w            700000 non-null  float64
 14  branch_req_count_8w    700000 non-null  int64  

* train과 test 모두 결측치가 없음

In [5]:
train.describe()

,id,fraud_bool,yearly_income,name_email_sim,prev_addr_months,curr_addr_months,age_bucket,days_since_req,init_transfer_amt,zip_req_count_4w,...,is_mobile_valid,bank_months_count,has_other_cards,req_credit_limit,is_foreign_req,session_length_min,is_session_persistent,device_email_cnt_8w,device_fraud_history,month_idx
count,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,7.000000e+05,700000.000000,700000.00000,...,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.0,700000.000000
mean,349999.500000,0.010917,0.562678,0.493689,16.719490,86.623213,33.681886,1.026138e+00,8.683809,1571.70295,...,0.889567,10.835533,0.223013,515.476271,0.025331,7.530962,0.576723,1.018341,0.0,3.290106
std,202072.738554,0.103913,0.290138,0.289164,44.049827,88.439563,12.025505,5.380082e+00,20.258896,1005.21414,...,0.313429,12.111838,0.416267,487.297338,0.157130,8.017990,0.494079,0.180798,0.0,2.209817
min,0.000000,0.000000,0.100000,0.000020,-1.000000,-1.000000,10.000000,4.036860e-09,-15.530555,1.00000,...,0.000000,-1.000000,0.000000,190.000000,0.000000,-1.000000,0.000000,-1.000000,0.0,0.000000
25%,174999.750000,0.000000,0.300000,0.225274,-1.000000,19.000000,20.000000,7.184887e-03,-1.180621,893.00000,...,1.000000,-1.000000,0.000000,200.000000,0.000000,3.101577,0.000000,1.000000,0.0,1.000000
50%,349999.500000,0.000000,0.600000,0.492128,-1.000000,52.000000,30.000000,1.517812e-02,-0.830318,1261.00000,...,1.000000,5.000000,0.000000,200.000000,0.000000,5.105845,1.000000,1.000000,0.0,3.000000
75%,524999.250000,0.000000,0.800000,0.755541,12.000000,130.000000,40.000000,2.633658e-02,5.147847,1943.00000,...,1.000000,25.000000,0.000000,500.000000,0.000000,8.849767,1.000000,1.000000,0.0,5.000000
max,699999.000000,1.000000,0.900000,0.999999,381.000000,425.000000,90.000000,7.845690e+01,112.452078,6700.00000,...,1.000000,32.000000,1.000000,2100.000000,1.000000,85.899143,1.000000,2.000000,0.0,7.000000


In [6]:
train.select_dtypes(include='str').columns

Index(['payment_type', 'employment_status', 'housing_status',
       'application_source', 'device_os'],
      dtype='str')

In [7]:
train['payment_type'].value_counts()

payment_type
AB    259361
AA    180991
AC    176293
AD     83149
AE       206
Name: count, dtype: int64

In [8]:
train['employment_status'].value_counts()

employment_status
CA    511680
CB     96618
CF     30768
CC     26337
CD     18489
CE     15796
CG       312
Name: count, dtype: int64

In [9]:
train['housing_status'].value_counts()

housing_status
BC    260351
BB    182903
BA    118769
BE    118324
BD     18318
BF      1159
BG       176
Name: count, dtype: int64

In [10]:
train['application_source'].value_counts()

application_source
INTERNET    695070
TELEAPP       4930
Name: count, dtype: int64

In [11]:
train['device_os'].value_counts()

device_os
other        240240
linux        233157
windows      183920
macintosh     37670
x11            5013
Name: count, dtype: int64

# Preprocessing

### 1. 결측치 처리

결측치 없음.

### 2. 범주형 데이터 인코딩

 payment_type

In [12]:
def group_payment_type(payment_type):
    # payment type: AB, AA, AC, AD, AE
    if payment_type == 'AA':
        return 0
    elif payment_type == 'AB':
        return 1
    elif payment_type == 'AC':
        return 2
    elif payment_type == 'AD':
        return 3
    elif payment_type == 'AE':
        return 4
    else:
        return -1
    
train['payment_type_group'] = train['payment_type'].apply(group_payment_type)


employment_status

In [13]:
def group_employment_status(employment_status):
    # employment stat: CA, CB, CC, CD, CE, CF, CG
    if employment_status == 'CA':
        return 0
    elif employment_status == 'CB':
        return 1
    elif employment_status == 'CC':
        return 2
    elif employment_status == 'CD':
        return 3
    elif employment_status == 'CE':
        return 4
    elif employment_status == 'CF':
        return 5
    elif employment_status == 'CG':
        return 6
    else:
        return -1

train['employment_status_group'] = train['employment_status'].apply(group_employment_status)

housing_status

In [14]:
def group_housing_status(housing_status):
    # housing stat: CL, CM, CN, CO, CP
    if housing_status == 'CL':
        return 0
    elif housing_status == 'CM':
        return 1
    elif housing_status == 'CN':
        return 2
    elif housing_status == 'CO':
        return 3
    elif housing_status == 'CP':
        return 4
    else:
        return -1
    
train['housing_status_group'] = train['housing_status'].apply(group_housing_status)

application_source

In [15]:
def group_application_source(application_source):
    # INTERNET, TELEAPP
    if application_source == 'INTERNET':
        return 0
    elif application_source == 'TELEAPP':
        return 1
    else:
        return -1
train['application_source_group'] = train['application_source'].apply(group_application_source)

device_os

In [16]:
def group_device_os(device_os):
    if device_os == 'other':
        return 0
    elif device_os == 'linux':
        return 1
    elif device_os == 'windows':
        return 2
    elif device_os == 'macintosh':
        return 3
    elif device_os == 'x11':
        return 4
    else:
        return -1
    
train['device_os_group'] = train['device_os'].apply(group_device_os)

In [18]:
train.drop(columns=['application_source', 'device_os', 'payment_type', 'employment_status', 'housing_status'], inplace=True)

# Feature Engineering

# Validation 전략

K-Fold CV 사용

# Baseline 모델

# Iteration 개선